# CrewAI in Google Colab → Local Agent via Greft

This notebook demonstrates a CrewAI **Software Release Analyst** running in Google Colab and sending its completed analysis to an independently running local Greft identity, `@terminal-agent`.

**Flow**

```text
CrewAI / @colab-agent
        ↓
Greft MCP (`send_message`)
        ↓
Greft relay
        ↓
@terminal-agent / local terminal
```

The issue data in this notebook is synthetic. No private project data is required.


## 1. Install dependencies

The versions are pinned so the example is reproducible. If Colab asks you to restart the runtime after installation, restart it and continue from the next cell.


In [6]:
%pip install -q "crewai[litellm]==1.15.22" "mcp"


## 2. Load secrets

Create these entries in the Colab **Secrets** panel before running this cell:

- `OPENROUTER_API_KEY`
- `GREFT_API_KEY`
- `GREFT_API_URL`

`GREFT_API_URL` should be the relay base URL, for example `https://relay.example.com`, without `/mcp`.

The cell validates configuration without printing secret values.


In [7]:
from google.colab import userdata

def get_secret(name: str) -> str:
    value = userdata.get(name)
    if not value:
        raise RuntimeError(f"Missing Colab secret: {name}")
    return value.strip()

OPENROUTER_API_KEY = get_secret("OPENROUTER_API_KEY")
GREFT_API_KEY = get_secret("GREFT_API_KEY")
GREFT_API_URL = get_secret("GREFT_API_URL").rstrip("/")

if not GREFT_API_URL.startswith("https://"):
    raise RuntimeError("GREFT_API_URL must use HTTPS for this remote Colab demo.")

COLAB_ADDRESS = "@ralph-planner"
TERMINAL_ADDRESS = "@reviewer"

print("Configuration loaded successfully.")


Configuration loaded successfully.


## 3. Configure OpenRouter and Greft MCP

The LLM uses OpenRouter's free-model router. The router can select a free model that supports the capabilities required by the request, including tool calling.

For safety, the CrewAI agent receives only Greft's `send_message` MCP tool.


In [8]:
from crewai import Agent, Crew, LLM, Task
from crewai.mcp import MCPServerHTTP
from crewai.mcp.filters import create_static_tool_filter

llm = LLM(
    model="openrouter/openrouter/free",
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
    max_tokens=1200,
)

greft_mcp = MCPServerHTTP(
    url=f"{GREFT_API_URL}/mcp?address={COLAB_ADDRESS}",
    headers={
        "Authorization": f"Bearer {GREFT_API_KEY}",
    },
    streamable=True,
    tool_filter=create_static_tool_filter(
        allowed_tool_names=["send_message"]
    ),
    cache_tools_list=True,
)

print("OpenRouter and Greft MCP configured.")


OpenRouter and Greft MCP configured.


## 4. Create the CrewAI agent

The agent's job is intentionally narrow: analyze release issues and hand the result to another independently running agent through Greft.


In [9]:
worker = Agent(
    role="Software Release Analyst",
    goal=(
        "Analyze software release issues, determine their priority, "
        "and hand the resulting assessment to another agent through Greft."
    ),
    backstory=(
        "You are a software release analyst running remotely inside "
        "Google Colab. You assess reported software issues and communicate "
        "your findings to another independently running agent using Greft."
    ),
    llm=llm,
    mcps=[greft_mcp],
    verbose=True,
)


## 5. Define the release-analysis task

Before running the task, start the receiving agent on your local machine:

```bash
greft connect
```


In [10]:
release_analysis_task = Task(
    description=f"""
You are reviewing a software release.

Analyze the following synthetic issue reports:

1. Login form accepts an invalid email address.
2. Dashboard crashes when the API returns an empty response.
3. Button spacing is slightly inconsistent on mobile devices.
4. Password reset links sometimes expire immediately after being generated.
5. The browser console contains a non-sensitive warning.

Your job is to:

1. Identify which issues are the most serious.
2. Classify each issue as High, Medium, or Low priority.
3. Briefly explain why each priority was chosen.
4. Recommend which issues should be fixed before the release.
5. Produce a concise final release assessment.

After completing the analysis, you MUST use the Greft `send_message`
tool to send the complete final analysis to:

{TERMINAL_ADDRESS}

Do not merely say that you would send the message.
Actually call the `send_message` tool.

Send only one final message to the terminal agent.
""",
    expected_output=f"""
A concise software release analysis containing:

- High-priority issues
- Medium-priority issues
- Low-priority issues
- Issues that should block the release
- A short final recommendation

The complete analysis must also be sent through Greft to
{TERMINAL_ADDRESS}.
""",
    agent=worker,
)


## 6. Run the crew

The task is successful when:

1. CrewAI completes the release analysis.
2. It calls Greft's `send_message` tool.
3. The complete analysis appears in the terminal listening as `@terminal-agent`.


In [11]:
crew = Crew(
    agents=[worker],
    tasks=[release_analysis_task],
    verbose=True,
)

result = await crew.kickoff_async()

print("\n--- CrewAI final result ---\n")
print(result)


╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: f13404b2-3b6e-4184-8ac1-eda1d1320bb8                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🔌 MCP Connection ───────────────────────────────────────────────╮
│                                                                                                                 │
│  MCP Connection Started                                                                                         │
│                                                                                                                 │
│  URL: https://greft-relay-783768789695.us-central1.run.app/mcp?address=@ralph-planner                           │
│  Transport: streamable-http                                                                                     │
│  Timeout: 30s                                                                                                   │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── ✅ MCP Connected ────────────────────────────────────────────────╮
│                                                                                                                 │
│  MCP Connection Completed                                                                                       │
│                                                                                                                 │
│  Server: greft-relay-783768789695.us-central1.run.app                                                           │
│  URL: https://greft-relay-783768789695.us-central1.run.app/mcp?address=@ralph-planner                           │
│  Transport: streamable-http                                                                                     │
│  Duration: 994.38ms                                                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name:                                                                                                          │
│  You are reviewing a software release.                                                                          │
│                                                                                                                 │
│  Analyze the following synthetic issue reports:                                                                 │
│                                                                                                                 │
│  1. Login form accepts an invalid email address.                                                                │
│  2. Dashboard crashes when the API returns an empty response.                                                   │
│  3. Button spacing is slightly inconsistent on mobile devices.                                                  │
│  4. Password reset links sometimes expire immediately after being generated.                                    │
│  5. The browser console contains a non-sensitive warning.                                                       │
│                                                                                                                 │
│  Your job is to:                                                                                                │
│                                                                                                                 │
│  1. Identify which issues are the most serious.                                                                 │
│  2. Classify each issue as High, Medium, or Low priority.                                                       │
│  3. Briefly explain why each priority was chosen.                                                               │
│  4. Recommend which issues should be fixed before the release.                                                  │
│  5. Produce a concise final release assessment.                                                                 │
│                                                                                                                 │
│  After completing the analysis, you MUST use the Greft `send_message`                                           │
│  tool to send the complete final analysis to:                                                                   │
│                                                                                                                 │
│  @reviewer                                                                                                      │
│                                                                                                                 │
│  Do not merely say that you would send the message.                                                             │
│  Actually call the `send_message` tool.                                                                         │
│                                                                                                                 │
│  Send only one final message to the terminal agent.                                                             │
│                                                                                                                 │
│  ID: 7f26359d-492a-4385-af04-670e72db45fc                                                                       │
│                                                                                                                 │
│                                                        

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Software Release Analyst                                                                                │
│                                                                                                                 │
│  Task:                                                                                                          │
│  You are reviewing a software release.                                                                          │
│                                                                                                                 │
│  Analyze the following synthetic issue reports:                                                                 │
│                                                                                                                 │
│  1. Login form accepts an invalid email address.                                                                │
│  2. Dashboard crashes when the API returns an empty response.                                                   │
│  3. Button spacing is slightly inconsistent on mobile devices.                                                  │
│  4. Password reset links sometimes expire immediately after being generated.                                    │
│  5. The browser console contains a non-sensitive warning.                                                       │
│                                                                                                                 │
│  Your job is to:                                                                                                │
│                                                                                                                 │
│  1. Identify which issues are the most serious.                                                                 │
│  2. Classify each issue as High, Medium, or Low priority.                                                       │
│  3. Briefly explain why each priority was chosen.                                                               │
│  4. Recommend which issues should be fixed before the release.                                                  │
│  5. Produce a concise final release assessment.                                                                 │
│                                                                                                                 │
│  After completing the analysis, you MUST use the Greft `send_message`                                           │
│  tool to send the complete final analysis to:                                                                   │
│                                                                                                                 │
│  @reviewer                                                                                                      │
│                                                                                                                 │
│  Do not merely say that you would send the message.                                                             │
│  Actually call the `send_message` tool.                                                                         │
│                                                                                                                 │
│  Send only one final message to the terminal agent.                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰────────────────────────────────────────────────────────

...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Software Release Analyst                                                                                │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
│  # Software Release Assessment                                                                                  │
│                                                                                                                 │
│  ## High-Priority Issues                                                                                        │
│                                                                                                                 │
│  **Issue 2: Dashboard crashes when the API returns an empty response.**                                         │
│  - **Severity:** Critical/High                                                                                  │
│  - **Rationale:** This causes application failure and a broken user experience at runtime. Empty responses are  │
│  a common API scenario that must be handled gracefully. Unhandled crashes can lead to user frustration, data    │
│  loss, or complete workflow disruption.                                                                         │
│                                                                                                                 │
│  **Issue 4: Password reset links sometimes expire immediately after being generated.**                          │
│  - **Severity:** High                                                                                           │
│  - **Rationale:** This directly impacts user security and account recovery workflows. Users locked out of       │
│  their accounts cannot reliably regain access, which degrades trust and can create support escalations.         │
│  Security and authentication defects should be treated as high priority.                                        │
│                                                                                                                 │
│  ## Medium-Priority Issues                                                                                      │
│                                                                                                                 │
│  **Issue 1: Login form accepts an invalid email address.**                                                      │
│  - **Severity:** Medium                                                                                         │
│  - **Rationale:** This is a validation defect that allows malformed or unintended input. While it may not       │
│  crash the system immediately, it can lead to downstream processing failures, incorrect data storage, or        │
│  failed communication delivery (e.g., notification emails). It degrades system reliability and should be fixed  │
│  before release, but does not constitute a critical failure on its own.                                         │
│                                                                                                                 │
│  **Issue 3: Button spacing is slightly inconsistent on mobile devices.**                                        │
│  - **Severity:** Medium                                                                                         │
│  - **Rationale:** This is a user interface quality issu

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name:                                                                                                          │
│  You are reviewing a software release.                                                                          │
│                                                                                                                 │
│  Analyze the following synthetic issue reports:                                                                 │
│                                                                                                                 │
│  1. Login form accepts an invalid email address.                                                                │
│  2. Dashboard crashes when the API returns an empty response.                                                   │
│  3. Button spacing is slightly inconsistent on mobile devices.                                                  │
│  4. Password reset links sometimes expire immediately after being generated.                                    │
│  5. The browser console contains a non-sensitive warning.                                                       │
│                                                                                                                 │
│  Your job is to:                                                                                                │
│                                                                                                                 │
│  1. Identify which issues are the most serious.                                                                 │
│  2. Classify each issue as High, Medium, or Low priority.                                                       │
│  3. Briefly explain why each priority was chosen.                                                               │
│  4. Recommend which issues should be fixed before the release.                                                  │
│  5. Produce a concise final release assessment.                                                                 │
│                                                                                                                 │
│  After completing the analysis, you MUST use the Greft `send_message`                                           │
│  tool to send the complete final analysis to:                                                                   │
│                                                                                                                 │
│  @reviewer                                                                                                      │
│                                                                                                                 │
│  Do not merely say that you would send the message.                                                             │
│  Actually call the `send_message` tool.                                                                         │
│                                                                                                                 │
│  Send only one final message to the terminal agent.                                                             │
│                                                                                                                 │
│  Agent: Software Release Analyst                                                                                │
│                                                                                                                 │
│                                                        

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: f13404b2-3b6e-4184-8ac1-eda1d1320bb8                                                                       │
│  Final Output:                                                                                                  │
│                                                                                                                 │
│  # Software Release Assessment                                                                                  │
│                                                                                                                 │
│  ## High-Priority Issues                                                                                        │
│                                                                                                                 │
│  **Issue 2: Dashboard crashes when the API returns an empty response.**                                         │
│  - **Severity:** Critical/High                                                                                  │
│  - **Rationale:** This causes application failure and a broken user experience at runtime. Empty responses are  │
│  a common API scenario that must be handled gracefully. Unhandled crashes can lead to user frustration, data    │
│  loss, or complete workflow disruption.                                                                         │
│                                                                                                                 │
│  **Issue 4: Password reset links sometimes expire immediately after being generated.**                          │
│  - **Severity:** High                                                                                           │
│  - **Rationale:** This directly impacts user security and account recovery workflows. Users locked out of       │
│  their accounts cannot reliably regain access, which degrades trust and can create support escalations.         │
│  Security and authentication defects should be treated as high priority.                                        │
│                                                                                                                 │
│  ## Medium-Priority Issues                                                                                      │
│                                                                                                                 │
│  **Issue 1: Login form accepts an invalid email address.**                                                      │
│  - **Severity:** Medium                                                                                         │
│  - **Rationale:** This is a validation defect that allows malformed or unintended input. While it may not       │
│  crash the system immediately, it can lead to downstream processing failures, incorrect data storage, or        │
│  failed communication delivery (e.g., notification emails). It degrades system reliability and should be fixed  │
│  before release, but does not constitute a critical failure on its own.                                         │
│                                                                                                                 │
│  **Issue 3: Button spacing is slightly inconsistent on mobile devices.**                                        │
│  - **Severity:** Medium                                                                                         │
│  - **Rationale:** This is a user interface quality iss


--- CrewAI final result ---



# Software Release Assessment

## High-Priority Issues

**Issue 2: Dashboard crashes when the API returns an empty response.**
- **Severity:** Critical/High
- **Rationale:** This causes application failure and a broken user experience at runtime. Empty responses are a common API scenario that must be handled gracefully. Unhandled crashes can lead to user frustration, data loss, or complete workflow disruption.

**Issue 4: Password reset links sometimes expire immediately after being generated.**
- **Severity:** High
- **Rationale:** This directly impacts user security and account recovery workflows. Users locked out of their accounts cannot reliably regain access, which degrades trust and can create support escalations. Security and authentication defects should be treated as high priority.

## Medium-Priority Issues

**Issue 1: Login form accepts an invalid email address.**
- **Severity:** Medium
- **Rationale:** This is a validation defect that allows 

## Expected result

Your local terminal should receive a message from `@colab-agent` containing the prioritized release assessment.

The wording can vary because the model is not deterministic. The important behavior is the handoff:

```text
Colab CrewAI agent
        ↓
release analysis
        ↓
Greft send_message
        ↓
@terminal-agent
```

### Before committing this notebook

Use **Edit → Clear all outputs** (or the equivalent Colab command) if any output contains account-specific information. Never commit API keys or bearer tokens.
